In [ ]:
!pip install -q transformers datasets peft accelerate trl bitsandbytes wandb

In [ ]:
import os
import gc
import torch
from datetime import datetime
import json
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    pipeline
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from trl import SFTTrainer
import wandb

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive')


In [ ]:
if IS_COLAB:
    PROJECT_DIR = "/content/drive/MyDrive/ecommerce_agent_project_repo"
    DATA_PATH = f"{PROJECT_DIR}/sft_full_merged_final.jsonl"
    OUTPUT_DIR = f"{PROJECT_DIR}/llama_finetuned_model"
    CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
    HF_MODEL_ID = "rri02/llama-3.1-8b-ecommerce-finetuned"
else:
    PROJECT_DIR = "."
    DATA_PATH = "data/sft_full_merged_final.jsonl"
    OUTPUT_DIR = "out/llama_finetuned_model"
    CHECKPOINT_DIR = "out/checkpoints"
    HF_MODEL_ID = "rri02/llama-3.1-8b-ecommerce-finetuned"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = ""
WANDB_API_KEY = ""

TRAINING_CONFIG = {
    "num_train_epochs": 2,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 1,
    "learning_rate": 1e-4,
    "warmup_ratio": 0.1,
    "logging_steps": 25,
    "save_steps": 1000,
    "eval_steps": 1000,
    "save_total_limit": 3,
    "eval_strategy": "steps",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "fp16": False,
    "bf16": True,
    "dataloader_pin_memory": False,
    "remove_unused_columns": False,
    "group_by_length": True,
    "max_grad_norm": 1.0,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.01,
    "report_to": "wandb",
    "run_name": f"llama-3.1-8b-ecommerce-{datetime.now().strftime('%Y%m%d-%H%M')}",
    "push_to_hub": True,
    "hub_model_id": HF_MODEL_ID,
    "hub_strategy": "checkpoint",
    "hub_token": HF_TOKEN,
    "save_strategy": "steps",
    "resume_from_checkpoint": True,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit",
}

LORA_CONFIG = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)


In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

import os
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

wandb.init(
    project="llama-3.1-8b-ecommerce-finetuning",
    name=TRAINING_CONFIG["run_name"],
    config={
        "model_name": MODEL_NAME,
        "dataset_path": DATA_PATH,
        "lora_config": LORA_CONFIG.__dict__,
        **TRAINING_CONFIG
    }
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, LORA_CONFIG)

In [ ]:
from transformers.trainer_utils import get_last_checkpoint
import glob, os

def load_jsonl_dataset(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

raw_data = load_jsonl_dataset(DATA_PATH)
print(f"Loaded {len(raw_data)} training examples")

dataset = Dataset.from_list(raw_data)

dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    **TRAINING_CONFIG,
    logging_dir=f"{PROJECT_DIR}/logs" if IS_COLAB else "out/logs",
)

try:
    last = get_last_checkpoint(CHECKPOINT_DIR)
    if last is None:
        ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")))
        last = ckpts[-1] if ckpts else None

    if last:
        print(f"Resuming training from: {last}")
        train_result = trainer.train(resume_from_checkpoint=last)
    else:
        train_result = trainer.train()

    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    trainer.push_to_hub()

except Exception as e:
    print(f"Training interrupted: {e}")

del model, trainer
gc.collect()
torch.cuda.empty_cache()